In [0]:
from pyspark.sql import functions as F

ORDERS_PATH = "/Volumes/workspace/ecommerce_dataset/ecommerce/orders_raw.csv"
CUSTOMERS_PATH = "/Volumes/workspace/ecommerce_dataset/ecommerce/customers_raw.csv"

orders_raw = spark.read.csv(ORDERS_PATH, header=True, inferSchema=True)
customers_raw = spark.read.csv(CUSTOMERS_PATH, header=True, inferSchema=True)

print("Raw datasets loaded successfully.")

In [0]:
order_row_count = orders_raw.count()
customer_row_count = customers_raw.count()

print("ORDERS DATASET")
print(f"Total records: {order_row_count:,}")
print(f"Columns: {orders_raw.columns}")

print("\nOrders schema:")
orders_raw.printSchema()

print("\nCUSTOMERS DATASET")
print(f"Total records: {customer_row_count:,}")
print(f"Columns: {customers_raw.columns}")

print("\nCustomers schema:")
customers_raw.printSchema()

In [0]:
print("Sample orders:")
display(orders_raw.limit(20))

print("Sample customers:")
display(customers_raw.limit(20))

In [0]:
orders_null_counts = orders_raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in orders_raw.columns
])

customers_null_counts = customers_raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in customers_raw.columns
])

print("NULL counts - Orders:")
display(orders_null_counts)

print("NULL counts - Customers:")
display(customers_null_counts)

In [0]:
print("ORDER NULL PERCENTAGES:")

for c in orders_raw.columns:
    null_count = orders_raw.filter(F.col(c).isNull()).count()
    if null_count > 0:
        percentage = (null_count / order_row_count) * 100
        print(f"{c}: {null_count:,} ({percentage:.2f}%)")

print("\nCUSTOMER NULL PERCENTAGES:")

for c in customers_raw.columns:
    null_count = customers_raw.filter(F.col(c).isNull()).count()
    if null_count > 0:
        percentage = (null_count / customer_row_count) * 100
        print(f"{c}: {null_count:,} ({percentage:.2f}%)")

In [0]:
exact_order_duplicates = orders_raw.groupBy(orders_raw.columns).count().filter(F.col("count") > 1)
exact_customer_duplicates = customers_raw.groupBy(customers_raw.columns).count().filter(F.col("count") > 1)

duplicate_order_ids = orders_raw.groupBy("order_id").count().filter(F.col("count") > 1).orderBy(F.desc("count"))
duplicate_customer_ids = customers_raw.groupBy("customer_id").count().filter(F.col("count") > 1).orderBy(F.desc("count"))

exact_order_duplicate_count = exact_order_duplicates.count()
duplicate_order_id_count = duplicate_order_ids.count()

print(f"Exact duplicated order records: {exact_order_duplicate_count:,}")
print(f"Order IDs appearing more than once: {duplicate_order_id_count:,}")

if duplicate_order_id_count > 0:
    display(duplicate_order_ids.limit(20))

exact_customer_duplicate_count = exact_customer_duplicates.count()
duplicate_customer_id_count = duplicate_customer_ids.count()

print(f"\nExact duplicated customer records: {exact_customer_duplicate_count:,}")
print(f"Customer IDs appearing more than once: {duplicate_customer_id_count:,}")

if duplicate_customer_id_count > 0:
    display(duplicate_customer_ids.limit(20))

In [0]:
orders_profile = (
    orders_raw
    .withColumn("_quantity_numeric", F.expr("try_cast(quantity as int)"))
    .withColumn("_unit_price_numeric", F.expr("try_cast(unit_price as double)"))
    .withColumn("_order_date_parsed", F.expr("try_cast(order_date as date)"))
)

invalid_quantity = orders_profile.filter(F.col("_quantity_numeric").isNull() | (F.col("_quantity_numeric") <= 0))
invalid_price = orders_profile.filter(F.col("_unit_price_numeric").isNull() | (F.col("_unit_price_numeric") <= 0))
invalid_date = orders_profile.filter(F.col("order_date").isNull() | F.col("_order_date_parsed").isNull())
missing_customer_id = orders_raw.filter(F.col("customer_id").isNull())

invalid_quantity_count = invalid_quantity.count()
invalid_price_count = invalid_price.count()
invalid_date_count = invalid_date.count()
missing_customer_id_count = missing_customer_id.count()

print(f"Invalid/missing quantities: {invalid_quantity_count:,}")
print(f"Invalid/missing unit prices: {invalid_price_count:,}")
print(f"Invalid/missing order dates: {invalid_date_count:,}")
print(f"Missing customer IDs: {missing_customer_id_count:,}")

if invalid_quantity_count > 0:
    print("\nSample invalid quantities:")
    display(invalid_quantity.select("order_id", "quantity", "unit_price", "status").limit(20))

if invalid_price_count > 0:
    print("\nSample invalid prices:")
    display(invalid_price.select("order_id", "quantity", "unit_price", "status").limit(20))

In [0]:
print("ORDER STATUS VALUES:")
status_distribution = orders_raw.groupBy("status").count().orderBy(F.desc("count"))
display(status_distribution)

print("PAYMENT METHOD VALUES:")
payment_distribution = orders_raw.groupBy("payment_method").count().orderBy(F.desc("count"))
display(payment_distribution)

print("CITY VALUES:")
order_city_distribution = orders_raw.groupBy("city").count().orderBy(F.desc("count"))
display(order_city_distribution)

print("PRODUCT CATEGORY VALUES:")
category_distribution = orders_raw.groupBy("product_category").count().orderBy(F.desc("count"))
display(category_distribution)

In [0]:
normalized_statuses = orders_raw.withColumn("_normalized_status", F.lower(F.trim(F.col("status"))))

unexpected_statuses = (
    normalized_statuses
    .filter(
        F.col("_normalized_status").isNull() |
        ~F.col("_normalized_status").isin(
            "complete", "completed",
            "pending",
            "cancel", "canceled", "cancelled",
            "refunded"
        )
    )
    .groupBy("_normalized_status")
    .count()
    .orderBy(F.desc("count"))
)

print("UNEXPECTED STATUS VALUES:")
display(unexpected_statuses)

In [0]:
print("CUSTOMER TYPE VALUES:")
customer_type_distribution = customers_raw.groupBy("customer_type").count().orderBy(F.desc("count"))
display(customer_type_distribution)

print("CUSTOMER CITY VALUES:")
customer_city_distribution = customers_raw.groupBy("city").count().orderBy(F.desc("count"))
display(customer_city_distribution)

missing_email_count = customers_raw.filter(F.col("email").isNull()).count()

malformed_emails = customers_raw.filter(
    F.col("email").isNotNull() &
    ~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
)

malformed_email_count = malformed_emails.count()

print(f"Missing emails: {missing_email_count:,}")
print(f"Malformed non-null emails: {malformed_email_count:,}")

if malformed_email_count > 0:
    display(malformed_emails.select("customer_id", "customer_name", "email").limit(20))

In [0]:
raw_order_customer_ids = orders_raw.filter(F.col("customer_id").isNotNull()).select("customer_id").distinct()
raw_customer_ids = customers_raw.filter(F.col("customer_id").isNotNull()).select("customer_id").distinct()

unmatched_customer_ids_raw = raw_order_customer_ids.join(raw_customer_ids, on="customer_id", how="left_anti")
unmatched_customer_id_count = unmatched_customer_ids_raw.count()

print(f"Customer IDs referenced by orders but missing from customer data: {unmatched_customer_id_count:,}")

if unmatched_customer_id_count > 0:
    display(unmatched_customer_ids_raw.limit(20))

In [0]:
print("\nORDERS")
print(f"Raw records: {order_row_count:,}")
print(f"Order IDs appearing more than once: {duplicate_order_id_count:,}")
print(f"Invalid/missing quantities: {invalid_quantity_count:,}")
print(f"Invalid/missing prices: {invalid_price_count:,}")
print(f"Invalid/missing dates: {invalid_date_count:,}")
print(f"Missing customer IDs: {missing_customer_id_count:,}")
print(f"Distinct raw status values: {status_distribution.count():,}")
print(f"Distinct raw payment methods: {payment_distribution.count():,}")

print("\nCUSTOMERS")
print(f"Raw records: {customer_row_count:,}")
print(f"Customer IDs appearing more than once: {duplicate_customer_id_count:,}")
print(f"Missing emails: {missing_email_count:,}")
print(f"Malformed non-null emails: {malformed_email_count:,}")
print(f"Distinct customer types: {customer_type_distribution.count():,}")

print("\nRELATIONSHIP QUALITY")
print(f"Order customer IDs not present in customer dataset: {unmatched_customer_id_count:,}")

print("\nObserved issues should be documented in reports/data_profile.md.")